# Post-Loan Default Risk Monitoring and SHAP Explainability

This notebook demonstrates:
- loading saved preprocessing artifacts
- inference using the finalized stacking ensemble
- probability-based risk prediction
- SHAP explainability analysis

## Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
import joblib


## Load Saved Models and Artifacts

In [2]:
model = joblib.load("final_stacking_model.pkl")
preprocess = joblib.load("preprocess_pipeline.pkl")
selected_feature_indices = joblib.load("selected_feature_indices.pkl")
raw_feature_columns = joblib.load("raw_feature_columns.pkl")

## Create Sample Borrower Input

In [3]:
sample_input = {
    "out_prncp": 12000,
    "out_prncp_inv": 11800,
    "total_rec_prncp": 3000,
    "total_rec_int": 1200,
    "last_pymnt_amnt": 450,
    "delinq_2yrs": 0,
    "mths_since_last_delinq": 999,
    "revol_bal": 8000,
    "revol_util": 35.0,
    "open_acc": 6,
    "total_acc": 18,
    "fico_range_low": 720,
    "fico_range_high": 724,
    "grade": "B",
    "sub_grade": "B2",
    "term": "36 months",
    "int_rate": 12.5
}

## Initialize Default Feature Values

In [4]:
# Get column names from preprocess pipeline
num_cols = preprocess.transformers_[0][2]
cat_cols = preprocess.transformers_[1][2]

print("Numeric columns:", len(num_cols))
print("Categorical columns:", len(cat_cols))

Numeric columns: 97
Categorical columns: 21


In [5]:
# Initialize defaults correctly
full_input = {}

# Assign default values for numerical features
for col in num_cols:
    full_input[col] = 0

# Assign placeholder values for categorical features
for col in cat_cols:
    full_input[col] = "UNKNOWN"

In [6]:
for key, value in sample_input.items():
    if key in full_input:
        full_input[key] = value

## Convert Input Dictionary to DataFrame

In [7]:
X_df = pd.DataFrame([full_input])

## Apply Preprocessing and Feature Selection

In [8]:
X_processed = preprocess.transform(X_df)
X_selected = X_processed[:, selected_feature_indices] # Select finalized training features

## Generate Default Risk Prediction

In [9]:
# Predict probability of loan default
prob_default = model.predict_proba(X_selected)[0, 1]
# Categorize borrower risk level
if prob_default < 0.3:
    risk = "Low Risk"
elif prob_default < 0.6:
    risk = "Medium Risk"
else:
    risk = "High Risk"

print("Default Probability:", round(prob_default, 4))
print("Risk Category:", risk)

Default Probability: 0.9955
Risk Category: High Risk
